In [1]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from scipy.sparse import load_npz
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader, TensorDataset


# ============================================================
# 1. CONFIGURATION
# ============================================================

from pathlib import Path

DATA_DIR = Path.cwd()
MODEL_DIR = Path("trained_rl_model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 1024
EPOCHS = 30
LEARNING_RATE = 0.001
PATIENCE = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 65)
print("REAL ESTATE RL VALUE MODEL TRAINING")
print("=" * 65)
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU. Training may take longer.")


# ============================================================
# 2. LOAD PREPARED DATA
# ============================================================

def load_matrix(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return load_npz(path).tocsr().astype(np.float32)


def load_target(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return np.load(path).astype(np.float32)


X_train = load_matrix("X_train.npz")
X_val = load_matrix("X_val.npz")

y_train = load_target("y_train.npy")
y_val = load_target("y_val.npy")

assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)
assert X_train.shape[1] == X_val.shape[1]

input_size = X_train.shape[1]

print("\nTraining samples :", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Input features    :", input_size)


# ============================================================
# 3. DEFINE NEURAL NETWORK
# ============================================================

class PropertyValueNetwork(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


model = PropertyValueNetwork(input_size).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

loss_function = nn.MSELoss()

print("\nModel architecture:")
print(model)


# ============================================================
# 4. MINI-BATCH TRAINING
# ============================================================

# The prepared matrices are sparse. Convert only each batch
# to dense form so we don't load all features into RAM at once.

def make_batch(X, y, indices):
    batch_x = X[indices].toarray().astype(
        np.float32, copy=False
    )
    batch_y = y[indices]

    return (
        torch.from_numpy(batch_x).to(device),
        torch.from_numpy(batch_y).to(device)
    )


def evaluate(X, y):
    model.eval()
    predictions = []

    with torch.no_grad():
        for start in range(0, X.shape[0], BATCH_SIZE):
            end = min(start + BATCH_SIZE, X.shape[0])
            indices = np.arange(start, end)

            xb, _ = make_batch(X, y, indices)
            pred = model(xb)

            predictions.append(
                pred.cpu().numpy()
            )

    predictions = np.concatenate(predictions)
    predictions = np.clip(predictions, 0.0, 1.0)

    mae = mean_absolute_error(y, predictions)
    rmse = np.sqrt(mean_squared_error(y, predictions))
    r2 = r2_score(y, predictions)

    return predictions, mae, rmse, r2


best_val_rmse = float("inf")
best_epoch = 0
epochs_without_improvement = 0

history = []

print("\nStarting training...\n")

for epoch in range(1, EPOCHS + 1):
    model.train()

    shuffled_indices = np.random.permutation(
        X_train.shape[0]
    )

    total_loss = 0.0
    processed = 0

    for start in range(
        0, X_train.shape[0], BATCH_SIZE
    ):
        indices = shuffled_indices[
            start:start + BATCH_SIZE
        ]

        xb, yb = make_batch(
            X_train, y_train, indices
        )

        optimizer.zero_grad(set_to_none=True)

        predictions = model(xb)
        loss = loss_function(predictions, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(indices)
        processed += len(indices)

    train_loss = total_loss / processed

    _, val_mae, val_rmse, val_r2 = evaluate(
        X_val, y_val
    )

    history.append({
        "epoch": epoch,
        "train_mse": train_loss,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "val_r2": val_r2
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train MSE: {train_loss:.5f} | "
        f"Val MAE: {val_mae:.5f} | "
        f"Val RMSE: {val_rmse:.5f} | "
        f"Val R2: {val_r2:.4f}"
    )

    # Save the best checkpoint based on validation RMSE.
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            MODEL_DIR / "best_model.pth"
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(
            f"\nEarly stopping: validation did not "
            f"improve for {PATIENCE} epochs."
        )
        break


# ============================================================
# 5. SAVE MODEL CONFIGURATION AND TRAINING HISTORY
# ============================================================

config = {
    "model_type": "contextual_bandit_reward_estimator",
    "architecture": [input_size, 256, 128, 64, 1],
    "output_activation": "sigmoid",
    "input_features": input_size,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs_requested": EPOCHS,
    "best_epoch": best_epoch,
    "best_validation_rmse": float(best_val_rmse),
    "random_seed": SEED
}

with open(MODEL_DIR / "model_config.json", "w") as f:
    json.dump(config, f, indent=4)

with open(MODEL_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=4)

print("\nBest model saved at epoch:", best_epoch)


# ============================================================
# 6. FINAL HELD-OUT TEST EVALUATION
# ============================================================

# Test data is used only after training and model selection.
X_test = load_matrix("X_test.npz")
y_test = load_target("y_test.npy")

checkpoint = torch.load(
    MODEL_DIR / "best_model.pth",
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint)

test_predictions, test_mae, test_rmse, test_r2 = evaluate(
    X_test, y_test
)

print("\n" + "=" * 65)
print("FINAL TEST RESULTS")
print("=" * 65)
print(f"Test MAE  : {test_mae:.5f}")
print(f"Test RMSE : {test_rmse:.5f}")
print(f"Test R2   : {test_r2:.4f}")

print("\nInterpretation:")
print("MAE  = average absolute reward prediction error.")
print("RMSE = prediction error with extra penalty for large errors.")
print("R2   = how much reward variation the model explains.")
print("Lower MAE/RMSE is better; R2 closer to 1 is better.")

# Save evaluation results
results = {
    "test_mae": float(test_mae),
    "test_rmse": float(test_rmse),
    "test_r2": float(test_r2),
    "best_epoch": best_epoch,
    "best_validation_rmse": float(best_val_rmse)
}

with open(MODEL_DIR / "evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nFILES CREATED:")
for filename in [
    "best_model.pth",
    "model_config.json",
    "training_history.json",
    "evaluation_results.json"
]:
    print(" -", MODEL_DIR / filename)

print("\nTRAINING AND EVALUATION COMPLETE.")
print("Next: build the Top 4 recommendation system.")

REAL ESTATE RL VALUE MODEL TRAINING
Device: cuda
GPU: NVIDIA H100 PCIe MIG 3g.40gb

Training samples : 400000
Validation samples: 50000
Input features    : 497

Model architecture:
PropertyValueNetwork(
  (network): Sequential(
    (0): Linear(in_features=497, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Linear(in_features=64, out_features=1, bias=True)
    (9): Sigmoid()
  )
)

Starting training...

Epoch 01/30 | Train MSE: 0.00857 | Val MAE: 0.02988 | Val RMSE: 0.04119 | Val R2: 0.9651
Epoch 02/30 | Train MSE: 0.00151 | Val MAE: 0.02029 | Val RMSE: 0.02900 | Val R2: 0.9827
Epoch 03/30 | Train MSE: 0.00090 | Val MAE: 0.01513 | Val RMSE: 0.02193 | Val R2: 0.9901
Epoch 04/30 | Train MSE: 0.00062 | Val MAE: 0.01277 | Val RMSE: 0.01828 | Val R2: 0.993